**Developing a simple chatbot using python and deep learning**

In [3]:
#%pip install nltk

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 558.9 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 558.9 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 588.4 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 588.4 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 588.4 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.6 MB 559.3 kB/s 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\oyelola Ibrahim\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [16]:
# import the libraries
import random
from tensorflow import keras
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
import numpy as np
import pickle
import json

In [17]:
#import nltk,WordNetLemmatizer
#download punkt and wordnet

import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
nltk.download("punkt")
nltk.download("wordnet")
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to C:\Users\oyelola
[nltk_data]     Ibrahim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\oyelola
[nltk_data]     Ibrahim\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\oyelola
[nltk_data]     Ibrahim\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [20]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to C:\Users\oyelola
[nltk_data]     Ibrahim\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [21]:
# init file
# create a list of words, classes and documents.
# open the intent.json file and read that file
words=[]
classes = []
documents = []
ignore_words = ['!', '?',]
data_file = open('intents.json').read()
intents = json.loads(data_file)

In [22]:
#iterate over intents
# now iterate over the patterns
# take each word and tokenize it

for intent in intents['intents']:
    for pattern in intent['patterns']:
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        documents.append((w, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [23]:
# lemmatizer
# first lower the words and then iterate through words and check if it is not present in the ignore words
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

In [24]:
#print the len of documents
print (len(documents), "documents")
print (len(classes), "classes", classes)
print (len(words), "unique lemmatized words", words)

100 documents
55 classes ['AI', 'abbr', 'artificial', 'bend', 'body', 'bot1', 'breathe', 'business', 'chatbot', 'chatterbox', 'clone', 'comp', 'computer', 'control', 'cramped', 'date', 'death', 'do', 'events', 'fav', 'fight', 'goodbye', 'greetings', 'hardware', 'hobby', 'idea', 'imortal', 'lang', 'laugh', 'lie', 'machine', 'malfunction', 'motormouth', 'move', 'name', 'name1', 'need', 'noanswer', 'os', 'program', 'programming', 'ratchet', 'robotics', 'robots', 'robotss', 'sapient', 'sense', 'sentiment', 'shoe', 'sound', 'stupid', 'thanks', 'usage', 'who', 'wt']
126 unique lemmatized words ["'m", "'s", ',', 'a', 'ai', 'all', 'allowed', 'am', 'an', 'are', 'artificial', 'awesome', 'be', 'being', 'bend', 'body', 'bot', 'breathe', 'business', 'bye', 'can', 'chat', 'chatterbox', 'clone', 'coffee', 'computer', 'control', 'cramped', 'data', 'date', 'die', 'do', 'entity', 'event', 'favorite', 'favour', 'fight', 'for', 'good', 'great', 'hardware', 'haroo', 'hello', 'help', 'helpful', 'helping', '

In [26]:
#create pickle dump for words and open with words.pkl and write in wb mode
pickle.dump(words,open('words.pkl','wb'))
#create pickle dump for classes and open with classes.pkl and write in wb mode
pickle.dump(classes,open('classes.pkl','wb'))

In [27]:
# training initializer
# inintializing training data
training = []
output_empty = [0] * len(classes)
for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)  
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    training.append([bag, output_row])

In [29]:
#shuffle our features and turn into np.array
random.shuffle(training)
training = np.array(training, dtype=object)
# create train and test lists. X - patterns, Y - intents
train_x = list(training[:,0])
train_y = list(training[:,1])
print("Training data created")

Training data created


In [32]:
# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons 
# equal to number of intents to predict output intent with softmax

#actual training
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

In [33]:
# check the summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │        16,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 55)             │         3,575 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,087 (109.71 KB)

 Trainable params: 28,087 (109.71 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
# Compile model. Stochastic gradient descent with Nesterov accelerated gradient gives good results for this model
sgd = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

C:\Users\oyelola Ibrahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [35]:
#fitting our model 
hist = model.fit(np.array(train_x), np.array(train_y), epochs=150, batch_size=5, verbose=1)
# save our model in .h5 extension
model.save('chatbot_model.h5', hist)
print("model created")

Epoch 1/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.0200 - loss: 4.0241    
Epoch 2/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0700 - loss: 3.9343   
Epoch 3/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0700 - loss: 3.8944   
Epoch 4/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1600 - loss: 3.7665
Epoch 5/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1400 - loss: 3.6965
Epoch 6/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1500 - loss: 3.6003
Epoch 7/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2100 - loss: 3.5457   
Epoch 8/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1800 - loss: 3.3892
Epoch 9/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2200 - loss: 3.3903
Epoch 10/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2700 - loss: 3.1886
Epoch 11/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2900 - loss: 3.1370
Epoch 12/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/st

model created


**Deployment**

In [37]:
%pip install flask_ngrok

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\oyelola Ibrahim\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
!jupyter nbconvert --to script Developing a simple chatbot using python and deep learning.ipynb

[NbConvertApp] WARNING | pattern 'Developing' matched no files
[NbConvertApp] WARNING | pattern 'a' matched no files
[NbConvertApp] WARNING | pattern 'simple' matched no files
[NbConvertApp] WARNING | pattern 'chatbot' matched no files
[NbConvertApp] WARNING | pattern 'using' matched no files
[NbConvertApp] WARNING | pattern 'and' matched no files
[NbConvertApp] WARNING | pattern 'deep' matched no files
[NbConvertApp] WARNING | pattern 'learning.ipynb' matched no files
[NbConvertApp] Converting notebook python to script
Traceback (most recent call last):
  File "C:\Users\oyelola Ibrahim\anaconda3\Lib\site-packages\nbformat\reader.py", line 20, in parse_json
    nb_dict = json.loads(s, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\oyelola Ibrahim\anaconda3\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\oyelola Ibrahim\anaconda3\Lib\json\decoder.py", line 337, in decode
    obj, end